# ROADGUARD — YOLOv8 License Plate Detection Training Pipeline
### AI-Powered Vehicle Document Verification System using ANPR

This notebook trains a custom YOLOv8 model for Indian License Plate localization on a **free Google Colab T4 GPU** in ~12 to 15 minutes.
- **Dataset**: 5,583 annotated images (Train: 3,960 | Val: 1,182 | Test: 441)
- **Model**: YOLOv8 Nano (`yolov8n.pt`)
- **Target Metric**: mAP@0.5 > 92%, Inference Latency < 25ms

## 1. Verify Free GPU Acceleration (Tesla T4)
Ensure `Runtime` -> `Change runtime type` is set to **T4 GPU**.

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

## 2. Install Dependencies

In [ ]:
!pip install -q ultralytics opencv-python matplotlib

## 3. Extract Dataset
Upload `unique_dataset.zip` directly to the Colab files panel on the left (or via Google Drive).

In [ ]:
import os
import zipfile
from pathlib import Path

if os.path.exists('unique_dataset.zip') and not os.path.exists('unique dataset'):
    print("Unzipping unique_dataset.zip...")
    with zipfile.ZipFile('unique_dataset.zip', 'r') as zip_ref:
        zip_ref.extractall('unique dataset')
    print("Unzip complete!")

DATASET_PATH = Path("unique dataset")
print(f"Dataset folder exists: {DATASET_PATH.exists()}")
print(f"Train images: {len(list((DATASET_PATH / 'train' / 'images').glob('*.*')))}")
print(f"Val images:   {len(list((DATASET_PATH / 'valid' / 'images').glob('*.*')))}")
print(f"Test images:  {len(list((DATASET_PATH / 'test' / 'images').glob('*.*')))}")

## 4. Train YOLOv8 Detector (~12 to 15 minutes on GPU)

In [ ]:
from ultralytics import YOLO

# Load base model weights
model = YOLO("yolov8n.pt")

# Train on 5,583 images
results = model.train(
    data="unique dataset/data.yaml",
    epochs=25,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=0.001,
    patience=10,
    device=0 if torch.cuda.is_available() else "cpu",
    project="ROADGUARD_Plate_Detector",
    name="yolov8n_custom_run",
    exist_ok=True
)

## 5. Evaluate on Unseen Test Dataset

In [ ]:
metrics = model.val(split="test")
print(f"=== ROADGUARD TEST EVALUATION RESULTS ===")
print(f"mAP@0.50:        {metrics.box.map50 * 100:.2f}%")
print(f"mAP@0.50:0.95:   {metrics.box.map * 100:.2f}%")
print(f"Precision (P):   {metrics.box.p[0] * 100:.2f}%")
print(f"Recall (R):      {metrics.box.r[0] * 100:.2f}%")

## 6. Display Training Convergence Graphs & Confusion Matrix

In [ ]:
from IPython.display import Image, display
from pathlib import Path

run_dir = Path("ROADGUARD_Plate_Detector/yolov8n_custom_run")
results_png = run_dir / "results.png"
if results_png.exists():
    print("Training Loss & Metric Curves:")
    display(Image(filename=str(results_png)))

conf_matrix = run_dir / "confusion_matrix.png"
if conf_matrix.exists():
    print("Confusion Matrix:")
    display(Image(filename=str(conf_matrix)))

## 7. Automatically Download `best.pt` & Results Package

In [ ]:
import shutil
from google.colab import files

# Package weights and plots into a zip file
shutil.make_archive('trained_model_results', 'zip', 'ROADGUARD_Plate_Detector/yolov8n_custom_run')
print("Downloading trained_model_results.zip (includes best.pt and all evaluation graphs)...")
files.download('trained_model_results.zip')